In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv('nlp_reviews_5000.csv')
data.head()

,review,result
0,"Honestly, Fantastic results. I would definitel...",positive
1,"To be fair, This was worse than I expected. I ...",negative
2,i am satisfied with the quality and overall pe...,positive
3,"In my opinion, Uncomfortable, poorly made, and...",negative
4,"Well, this was worse than i expected. i do not...",negative


In [3]:
data.isnull().sum()

,0
review,0
result,0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   review  5000 non-null   object
 1   result  5000 non-null   object
dtypes: object(2)
memory usage: 78.3+ KB


In [5]:
data.shape

(5000, 2)

In [13]:
data = data.map(lambda x: x.lower() if isinstance(x, str) else x)
data.head()

,review,result
0,"honestly, fantastic results. i would definitel...",positive
1,"to be fair, this was worse than i expected. i ...",negative
2,i am satisfied with the quality and overall pe...,positive
3,"in my opinion, uncomfortable, poorly made, and...",negative
4,"well, this was worse than i expected. i do not...",negative


In [19]:
import string

punc = f'[{string.punctuation}]'

data = data.apply(lambda X: X.str.replace(punc,'',regex=True) if X.dtype=='object' else X)



In [20]:
data

,review,result
0,honestly fantastic results i would definitely ...,positive
1,to be fair this was worse than i expected i do...,negative
2,i am satisfied with the quality and overall pe...,positive
3,in my opinion uncomfortable poorly made and un...,negative
4,well this was worse than i expected i do not r...,negative
...,...,...
4995,honestly i may use it again after trying it fo...,neutral
4996,for me the food tasted stale and unpleasant i ...,negative
4997,in my opinion the results were mixed with both...,neutral
4998,in my opinion i may use it again after trying ...,neutral


In [22]:
!pip install symspellpy pandas tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 19.4 MB/s  0:00:00


In [24]:
import pkg_resources
from symspellpy import SymSpell
from tqdm import tqdm


# 2. Initialize SymSpell
# max_dictionary_edit_distance=2 allows correcting up to 2 typos per word
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

# Load the built-in English frequency dictionary included with the library
dictionary_path = pkg_resources.resource_filename(
    "symspellpy", "frequency_dictionary_en_82_765.txt"
)
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)


# 3. Define the fast correction function
def correct_text(text):
    if not isinstance(text, str):
        return text  # Handle missing or NaN values safely

    # lookup_compound handles full sentences/phrases seamlessly
    suggestions = sym_spell.lookup_compound(text, max_edit_distance=2)

    # Return the best match term discovered by the algorithm
    return suggestions[0].term


# 4. Apply the correction to the entire column with a progress bar
tqdm.pandas(desc="Correcting Typos")
data["cleaned_review"] = data["review"].progress_apply(correct_text)

# 5. View your results
print(data[["review", "cleaned_review"]])


Correcting Typos: 100%|██████████| 5000/5000 [00:10<00:00, 482.19it/s]

                                                 review  \
0     honestly fantastic results i would definitely ...   
1     to be fair this was worse than i expected i do...   
2     i am satisfied with the quality and overall pe...   
3     in my opinion uncomfortable poorly made and un...   
4     well this was worse than i expected i do not r...   
...                                                 ...   
4995  honestly i may use it again after trying it fo...   
4996  for me the food tasted stale and unpleasant i ...   
4997  in my opinion the results were mixed with both...   
4998  in my opinion i may use it again after trying ...   
4999  honestly the app is simple to use and saves me...   

                                         cleaned_review  
0     honestly fantastic results i would definitely ...  
1     to be fair this was worse than i expected i do...  
2     i am satisfied with the quality and overall pe...  
3     in my opinion uncomfortable poorly made and un...  
4

In [25]:
import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')

data['tokens'] = data['cleaned_review'].astype(str).apply(word_tokenize)
data['tokens']

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


,tokens
0,"[honestly, fantastic, results, i, would, defin..."
1,"[to, be, fair, this, was, worse, than, i, expe..."
2,"[i, am, satisfied, with, the, quality, and, ov..."
3,"[in, my, opinion, uncomfortable, poorly, made,..."
4,"[well, this, was, worse, than, i, expected, i,..."
...,...
4995,"[honestly, i, may, use, it, again, after, tryi..."
4996,"[for, me, the, food, tasted, stale, and, unple..."
4997,"[in, my, opinion, the, results, were, mixed, w..."
4998,"[in, my, opinion, i, may, use, it, again, afte..."


In [30]:
from nltk.stem import PorterStemmer,SnowballStemmer

porter = PorterStemmer()
snowball = SnowballStemmer('english')

data['PorterS'] = data['tokens'].apply(lambda token_list :[porter.stem(word) for word in token_list])
data['SnowballS'] = data['tokens'].apply(lambda token_list : [snowball.stem(word)for word in token_list])

In [35]:
data[['tokens','PorterS','SnowballS']]

,tokens,PorterS,SnowballS
0,"[honestly, fantastic, results, i, would, defin...","[honestli, fantast, result, i, would, definit,...","[honest, fantast, result, i, would, definit, b..."
1,"[to, be, fair, this, was, worse, than, i, expe...","[to, be, fair, thi, wa, wors, than, i, expect,...","[to, be, fair, this, was, wors, than, i, expec..."
2,"[i, am, satisfied, with, the, quality, and, ov...","[i, am, satisfi, with, the, qualiti, and, over...","[i, am, satisfi, with, the, qualiti, and, over..."
3,"[in, my, opinion, uncomfortable, poorly, made,...","[in, my, opinion, uncomfort, poorli, made, and...","[in, my, opinion, uncomfort, poor, made, and, ..."
4,"[well, this, was, worse, than, i, expected, i,...","[well, thi, wa, wors, than, i, expect, i, do, ...","[well, this, was, wors, than, i, expect, i, do..."
...,...,...,...
4995,"[honestly, i, may, use, it, again, after, tryi...","[honestli, i, may, use, it, again, after, tri,...","[honest, i, may, use, it, again, after, tri, i..."
4996,"[for, me, the, food, tasted, stale, and, unple...","[for, me, the, food, tast, stale, and, unpleas...","[for, me, the, food, tast, stale, and, unpleas..."
4997,"[in, my, opinion, the, results, were, mixed, w...","[in, my, opinion, the, result, were, mix, with...","[in, my, opinion, the, result, were, mix, with..."
4998,"[in, my, opinion, i, may, use, it, again, afte...","[in, my, opinion, i, may, use, it, again, afte...","[in, my, opinion, i, may, use, it, again, afte..."


In [43]:
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')
lemmitizor = WordNetLemmatizer()

def lemmitization(token_list):
  if isinstance(token_list,list):
    return [lemmitizor.lemmatize(word,'v') for word in token_list]
  return token_list

data['Lemmitization'] = data['tokens'].apply(lemmitization)
data['Lemmitization']

data['Lemmitization'] = data['Lemmitization'].apply(lambda X:' '.join(X))

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [44]:
data['Lemmitization']

,Lemmitization
0,honestly fantastic result i would definitely b...
1,to be fair this be worse than i expect i do no...
2,i be satisfy with the quality and overall perf...
3,in my opinion uncomfortable poorly make and un...
4,well this be worse than i expect i do not reco...
...,...
4995,honestly i may use it again after try it for a...
4996,for me the food taste stale and unpleasant i c...
4997,in my opinion the result be mix with both stre...
4998,in my opinion i may use it again after try it ...


In [45]:
data.head(1)

,review,result,cleaned_review,tokens,PorterS,SnowballS,Lemmitization
0,honestly fantastic results i would definitely ...,positive,honestly fantastic results i would definitely ...,"[honestly, fantastic, results, i, would, defin...","[honestli, fantast, result, i, would, definit,...","[honest, fantast, result, i, would, definit, b...",honestly fantastic result i would definitely b...


In [46]:
data[['Lemmitization','result']]

,Lemmitization,result
0,honestly fantastic result i would definitely b...,positive
1,to be fair this be worse than i expect i do no...,negative
2,i be satisfy with the quality and overall perf...,positive
3,in my opinion uncomfortable poorly make and un...,negative
4,well this be worse than i expect i do not reco...,negative
...,...,...
4995,honestly i may use it again after try it for a...,neutral
4996,for me the food taste stale and unpleasant i c...,negative
4997,in my opinion the result be mix with both stre...,neutral
4998,in my opinion i may use it again after try it ...,neutral


In [48]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer =CountVectorizer()

bow_matrix = vectorizer.fit_transform(data['Lemmitization'])
vocabulary = vectorizer.get_feature_names_out()
bow_df = pd.DataFrame(bow_matrix.toarray(),columns=vocabulary)
bow_df

,about,absolutely,after,again,amaze,and,answer,app,arrive,as,...,waste,weaknesses,well,what,with,wonderful,work,worse,worth,would
0,0,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4995,0,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4996,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4997,0,0,0,0,0,1,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
4998,0,0,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
